# 00 · Data Cleaning (initial process)

**PhilGEPS procurement-failure risk · Notebook 0 of 3**
**Input:** the raw quarterly CSVs in `data/raw/` from 2023 Q1 to 2026 Q2 (28 files; the 2026 Q3 files are skipped).
**Output:** `data/raw/cleaned_data/PhilGeps_Cleaned.csv`, one row per bid line item. This is the main dataset of the project.
**Next step:** `python scripts/harmonise_cleaned.py` (adds `Source System`, merges the remaining 2026 column names, parses dates and amounts), then notebooks 01–03.

The cleaned CSV is committed with Git LFS, so `git lfs pull` downloads it and this notebook only needs re-running if the raw data changes. Memory: it holds all 28 files (about 11 million rows) in memory at once, so it needs a machine with well over 16 GB of RAM.

## DATA PREPROCESSING

### 1. READING ALL RAW DATA FILES
- **Data Scope:** Both versions (1.5 and 2.0), covering 2023 Q1 to 2026 Q2.
- Read all CSV files, accounting for different delimiters (comma-separated and tab-separated) and encoding formats.

### 2. COLUMN HEADER STANDARDIZATION
The following column names in Version 2.0 files were updated to match the corresponding column names in Version 1.5 files:

| Version 2.0 (Original) | Standardized Column Name |
|---|---|
| Delivery Period/Contract Duration | Delivery Period |
| Approved Budget for the Contract (ABC) | Approved Budget of the Contract |
| Awardee Size (MSME Category) | Awardee Size |
| Item Code | Line Item No |
| Item/Lot Description | Item Description |
| Item/Lot Name | Item Name |
| Published Date (Award) | Published Date(Award) |
| Unit of Measure | UOM |

### 3. MERGING OF DATASETS
- Concatenate all standardized datasets into a single DataFrame.
- Align records based on standardized column headers, retaining columns present in either version.
- Missing values in columns not available in a particular dataset are retained as null (`NaN`).

### 4. DUPLICATE RECORD REMOVAL
- Identify and remove exact duplicate rows across the combined dataset.
- Retain one occurrence of each identical record.

### 5. REMOVAL OF CLOSED-STATUS BIDS
- Filter out records with a **Closed** bid notice status.
- Retain the remaining records for subsequent exploratory data analysis and modelling.

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import os
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

# Set paths (repository root = first parent folder that contains data/raw)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").is_dir())
path = str(ROOT / "data" / "raw")
OUTPUT = ROOT / "data" / "raw" / "cleaned_data" / "PhilGeps_Cleaned.csv"
print(f"Data path: {path}")

1. READING ALL RAW DATA FILES

Data Scope : Both 1.5 and 2.0 version files from years 2023 Q1 - 2026 Q2

In [24]:
# Find all CSV files in the study window (2023 Q1 to 2026 Q2); file names are <system>_<year>_Q<quarter>.csv
def in_study_window(filename):
    _, year, quarter = Path(filename).stem.split("_")
    return (int(year), int(quarter[1])) <= (2026, 2)

csv_files = sorted([f for f in os.listdir(path) if f.lower().endswith(".csv") and in_study_window(f)])
print(f"Found {len(csv_files)} CSV files:\n")

for i, f in enumerate(csv_files, 1):
    print(f"{i}. {f}")

# Read all CSV files
dataframes = {}
file_info = []

print("\n" + "=" * 80)
print("Loading files...\n")

tab_delimited_files = {"1.5_2025_Q4.csv", "2.0_2025_Q4.csv"}

for file in csv_files:
    filepath = os.path.join(path, file)
    read_options = {"sep": "\t"} if file in tab_delimited_files else {}

    try:
        try:
            df = pd.read_csv(filepath, encoding="utf-8-sig", **read_options)
            encoding_used = "utf-8-sig"
        except UnicodeDecodeError:
            fallback = "latin-1" if file in tab_delimited_files else "cp1252"
            df = pd.read_csv(filepath, encoding=fallback, **read_options)
            encoding_used = fallback

        dataframes[file] = df
        file_info.append({
            "File": file,
            "Rows": len(df),
            "Columns": len(df.columns),
            "Size (MB)": round(os.path.getsize(filepath) / (1024 * 1024), 2),
            "Encoding": encoding_used,
        })
        print(f"✓ {file}: {len(df):,} rows, {len(df.columns)} columns ({encoding_used})")

    except Exception as e:
        print(f"✗ {file}: Error - {e}")

print(f"\n✓ Successfully loaded {len(dataframes)} of {len(csv_files)} files")

Found 28 CSV files:

1. 1.5_2023_Q1.csv
2. 1.5_2023_Q2.csv
3. 1.5_2023_Q3.csv
4. 1.5_2023_Q4.csv
5. 1.5_2024_Q1.csv
6. 1.5_2024_Q2.csv
7. 1.5_2024_Q3.csv
8. 1.5_2024_Q4.csv
9. 1.5_2025_Q1.csv
10. 1.5_2025_Q2.csv
11. 1.5_2025_Q3.csv
12. 1.5_2025_Q4.csv
13. 1.5_2026_Q1.csv
14. 1.5_2026_Q2.csv
15. 2.0_2023_Q1.csv
16. 2.0_2023_Q2.csv
17. 2.0_2023_Q3.csv
18. 2.0_2023_Q4.csv
19. 2.0_2024_Q1.csv
20. 2.0_2024_Q2.csv
21. 2.0_2024_Q3.csv
22. 2.0_2024_Q4.csv
23. 2.0_2025_Q1.csv
24. 2.0_2025_Q2.csv
25. 2.0_2025_Q3.csv
26. 2.0_2025_Q4.csv
27. 2.0_2026_Q1.csv
28. 2.0_2026_Q2.csv

Loading files...

✓ 1.5_2023_Q1.csv: 405,796 rows, 46 columns (utf-8-sig)
✓ 1.5_2023_Q2.csv: 436,736 rows, 46 columns (utf-8-sig)
✓ 1.5_2023_Q3.csv: 450,701 rows, 46 columns (utf-8-sig)
✓ 1.5_2023_Q4.csv: 450,311 rows, 46 columns (utf-8-sig)
✓ 1.5_2024_Q1.csv: 440,342 rows, 46 columns (utf-8-sig)
✓ 1.5_2024_Q2.csv: 474,381 rows, 46 columns (utf-8-sig)
✓ 1.5_2024_Q3.csv: 469,108 rows, 46 columns (utf-8-sig)
✓ 1.5_2024_Q4.csv

In [25]:
# CORRECTION: Tab delimited Q4 files
# Fix to correct value offset in these files. 

# Correct the one-column header offset in both Q4 files.
# Values stay in place; headers from Calendar Type onward move left by one.
q4_files_to_correct = {
    "1.5_2025_Q4.csv",
    "2.0_2025_Q4.csv",
}

for filename in q4_files_to_correct:
    if filename not in dataframes:
        print(f"{filename}: not loaded; skipped")
        continue

    df = dataframes[filename].copy()
    columns = list(df.columns)
    offset = columns.index("Calendar Type")

    corrected_columns = (
        columns[:offset]
        + columns[offset + 1:]
        + ["__extra_trailing"]
    )

    df.columns = corrected_columns
    df = df.drop(columns="__extra_trailing")

    dataframes[filename] = df
    print(f"Corrected shifted headers in {filename}")

Corrected shifted headers in 2.0_2025_Q4.csv
Corrected shifted headers in 1.5_2025_Q4.csv


2. COLUMN HARMONIZATION



In [27]:
# Gather all unique column headers for each version
headers_by_version = {"1.5": set(), "2.0": set()}

for filename, df in dataframes.items():
    version = filename.split("_", 1)[0]
    if version in headers_by_version:
        headers_by_version[version].update(df.columns)

headers_15 = headers_by_version["1.5"]
headers_20 = headers_by_version["2.0"]

print("\n" + "=" * 80)
print("COLUMN HEADER COMPARISON: 1.5 vs 2.0")
print("=" * 80)

print(f"\nHeaders in 1.5 files ({len(headers_15)}):")
print(*sorted(headers_15), sep="\n- ")

print(f"\nHeaders in 2.0 files ({len(headers_20)}):")
print(*sorted(headers_20), sep="\n- ")

print(f"\nShared headers ({len(headers_15 & headers_20)}):")
print(*sorted(headers_15 & headers_20), sep="\n- ")

print(f"\nOnly in 1.5 ({len(headers_15 - headers_20)}):")
print(*sorted(headers_15 - headers_20), sep="\n- ")

print(f"\nOnly in 2.0 ({len(headers_20 - headers_15)}):")
print(*sorted(headers_20 - headers_15), sep="\n- ")


COLUMN HEADER COMPARISON: 1.5 vs 2.0

Headers in 1.5 files (53):
Applicable Procurement Rules
- Approved Budget of the Contract
- Area of Delivery
- Area of Delivery/Project Location
- Award Date
- Award Notice Status
- Award Reference No.
- Award Title
- Awardee Joint Venture
- Awardee OrgID
- Awardee Organization Name
- Awardee Size
- Bid Notice Status
- Bid Reference No.
- Business Category
- Calendar Type
- City/Municipality
- City/Municipality of Awardee
- Classification
- Closing Date
- Contract Amount
- Contract Duration
- Contract Effectivity Date
- Contract End Date
- Country of Awardee
- Delivery Period
- Funding Instrument
- Funding Source
- Government Branch
- Item Budget
- Item Description
- Item Name
- Line Item No
- Notice Title
- Notice to Proceed Date
- PE Organization Type
- PE Organization Type (Grouped)
- Procurement Category
- Procurement Mode
- Procuring Entity (PE)
- Province
- Province of Awardee
- Published Date
- Published Date(Award)
- Quantity
- Region
- Re

2. RENAMING DATA HEADERS   

Based on above:

Delivery Period/Contract Duration ---> Delivery Period  
Approved Budget for the Contract (ABC)---> Approved Budget of the  Contract
Awardee Size (MSME Category) ----> Awardee Size  
Item Code ---> Line Item No  
Item/Lot Description---> Item Description  
Item/Lot Name----> Item Name  
Published Date (Award) ---> Published Date(Award)  
Unit of Measure ---> UOM  

In [28]:
# Rename 2.0 column headers to match the 1.5 naming
rename_20 = {
    "Delivery Period/Contract Duration": "Delivery Period",
    "Approved Budget for the Contract (ABC)": "Approved Budget of the Contract",
    "Awardee Size (MSME Category)": "Awardee Size",
    "Item Code": "Line Item No",
    "Item/Lot Description": "Item Description",
    "Item/Lot Name": "Item Name",
    "Published Date (Award)": "Published Date(Award)",
    "Unit of Measure": "UOM",
}

print("\n" + "=" * 80)
print("RENAMING 2.0 COLUMN HEADERS")
print("=" * 80)

for filename, df in dataframes.items():
    if filename.startswith("2.0_"):
        # Rename only headers that are present in this file
        applicable_renames = {
            old_name: new_name
            for old_name, new_name in rename_20.items()
            if old_name in df.columns
        }

        df.rename(columns=applicable_renames, inplace=True)

        if applicable_renames:
            print(f"\n{filename}:")
            for old_name, new_name in applicable_renames.items():
                print(f"  {old_name} → {new_name}")
        else:
            print(f"\n{filename}: no listed headers needed renaming")


RENAMING 2.0 COLUMN HEADERS

2.0_2023_Q1.csv: no listed headers needed renaming

2.0_2023_Q2.csv: no listed headers needed renaming

2.0_2023_Q3.csv: no listed headers needed renaming

2.0_2023_Q4.csv: no listed headers needed renaming

2.0_2024_Q1.csv: no listed headers needed renaming

2.0_2024_Q2.csv: no listed headers needed renaming

2.0_2024_Q3.csv: no listed headers needed renaming

2.0_2024_Q4.csv: no listed headers needed renaming

2.0_2025_Q1.csv: no listed headers needed renaming

2.0_2025_Q2.csv: no listed headers needed renaming

2.0_2025_Q3.csv: no listed headers needed renaming

2.0_2025_Q4.csv: no listed headers needed renaming

2.0_2026_Q1.csv:
  Delivery Period/Contract Duration → Delivery Period
  Approved Budget for the Contract (ABC) → Approved Budget of the Contract
  Awardee Size (MSME Category) → Awardee Size
  Item Code → Line Item No
  Item/Lot Description → Item Description
  Item/Lot Name → Item Name
  Published Date (Award) → Published Date(Award)
  Unit o

2. MERGING DATA INTO ONE DATASET

In [29]:
# Merge datasets
merged_df = pd.concat(dataframes.values(), ignore_index=True, sort=False)
print("\n" + "=" * 80)
print("MERGED DATASET SUMMARY")
print("=" * 80)
print(f"Rows: {len(merged_df):,}")
print(f"Columns: {len(merged_df.columns):,}")


MERGED DATASET SUMMARY
Rows: 10,976,576
Columns: 56


4. DROPPING DUPLICATES

In [30]:
# Drop exact duplicate rows and report the count
rows_before_duplicates = len(merged_df)
duplicate_count = int(merged_df.duplicated().sum())
merged_df_clean = merged_df.drop_duplicates().copy()
rows_after_duplicates = len(merged_df_clean)

print("\n" + "=" * 80)
print("DUPLICATE REMOVAL")
print("=" * 80)
print(f"Rows before: {rows_before_duplicates:,}")
print(f"Duplicate rows dropped: {duplicate_count:,}")
print(f"Rows after: {rows_after_duplicates:,}")


DUPLICATE REMOVAL
Rows before: 10,976,576
Duplicate rows dropped: 5,868,193
Rows after: 5,108,383


Reasons for duplicates

Overlapping reporting periods: The same procurement records may appear in multiple quarterly downloads.

Different dataset versions: Version 1.5 and 2.0 might contain many of the same records.

Repeated snapshots: The portal may have exported the same records repeatedly.

Actual duplicate records: The source itself may contain repeated identical rows.

5. DROPPING CLOSED BIDS

In [32]:
merged_df_clean["Bid Notice Status"].value_counts(normalize=True)

Bid Notice Status
Closed             0.590241
Awarded            0.341126
Failed             0.031673
Cancelled          0.031074
Active             0.005666
Shortlisted        0.000183
CONCLUDED          0.000034
CANCEL_APPROVED    0.000002
Published          0.000001
Name: proportion, dtype: float64

In [33]:
# Drop rows whose Bid Notice Status is Closed (case-insensitive)
if "Bid Notice Status" not in merged_df_clean.columns:
    raise KeyError("Column 'Bid Notice Status' was not found after loading.")

closed_mask = (
    merged_df_clean["Bid Notice Status"]
    .astype("string")
    .str.strip()
    .str.casefold()
    .eq("closed")
    .fillna(False)
)

closed_count = int(closed_mask.sum())
merged_df_clean = merged_df_clean.loc[~closed_mask].copy()

print("\n" + "=" * 80)
print("CLOSED STATUS REMOVAL")
print("=" * 80)
print(f"Closed-status rows dropped: {closed_count:,}")
print(f"Rows remaining: {len(merged_df_clean):,}")


CLOSED STATUS REMOVAL
Closed-status rows dropped: 3,015,173
Rows remaining: 2,093,210


In [36]:
# Save the Cleaned DataFrame
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
merged_df_clean.to_csv(OUTPUT, index=False)
print(f"Saved {len(merged_df_clean):,} rows to {OUTPUT}")

# Appendix 

1. Validation Checks

In [26]:
# Investigate unexpected Bid Notice Status values by source file
expected_statuses = {
    "active",
    "closed",
    "awarded",
    "cancelled",
    "failed",
    "concluded",
    "published",
}

for filename, df in dataframes.items():
    if "Bid Notice Status" not in df.columns:
        print(f"\n{filename}: Bid Notice Status column not found")
        continue

    status = df["Bid Notice Status"].astype("string").str.strip()
    normalized = status.str.casefold()

    unexpected_mask = status.notna() & ~normalized.isin(expected_statuses)
    unexpected = status[unexpected_mask]

    print(f"\n{'=' * 80}")
    print(filename)
    print(f"Rows: {len(df):,}")
    print(f"Unexpected non-empty status values: {int(unexpected_mask.sum()):,}")

    if not unexpected.empty:
        print("\nMost common unexpected values:")
        print(unexpected.value_counts().head(20).to_string())

        # Show the status and nearby columns to help identify shifted data
        context_cols = [
            col for col in [
                "Bid Reference No.",
                "Line Item No",
                "Item Name",
                "Bid Notice Status",
                "Award Reference No.",
                "Award Title",
                "UNSPSC Code",
            ]
            if col in df.columns
        ]

        print("\nExample rows with unexpected values:")
        print(df.loc[unexpected_mask, context_cols].head(10).to_string(index=False))


1.5_2023_Q1.csv
Rows: 405,796
Unexpected non-empty status values: 47

Most common unexpected values:
Bid Notice Status
Shortlisted    47

Example rows with unexpected values:
 Bid Reference No.  Line Item No                                          Item Name Bid Notice Status  Award Reference No. Award Title  UNSPSC Code
           9393803           1.0      Engagement of a Professional Service Provider       Shortlisted                  NaN         NaN          NaN
           9406798           1.0                                           23CSIA02       Shortlisted                  NaN         NaN          NaN
           9406854           1.0                                           23CSIA03       Shortlisted                  NaN         NaN          NaN
           9417235           1.0                                           23CSBE01       Shortlisted                  NaN         NaN          NaN
           9417282           1.0                                           23CSBE02 

In [20]:
for filename in ["1.5_2025_Q4.csv", "2.0_2025_Q4.csv"]:
    df = dataframes.get(filename)
    if df is None or "Bid Notice Status" not in df.columns:
        print(f"\n{filename}: not loaded or no Bid Notice Status column")
        continue

    status = df["Bid Notice Status"].astype("string").str.strip()
    bad_mask = status.str.fullmatch(r"\d+(?:\.0)?", na=False)

    print(f"\n{'=' * 80}\n{filename}")
    print(f"Rows with numeric Bid Notice Status: {int(bad_mask.sum()):,}")

    if bad_mask.any():
        row_index = bad_mask[bad_mask].index[0]
        row = df.loc[row_index]

        # Show every column and value for the first affected row
        for position, (column, value) in enumerate(row.items()):
            print(f"{position:>2}: {column:<40} {value}")


1.5_2025_Q4.csv
Rows with numeric Bid Notice Status: 1,854
 0: Procuring Entity (PE)                    MUNICIPALITY OF SANTA MARIA, ILOCOS SUR
 1: Region                                   Region I
 2: Province                                 Ilocos Sur
 3: City/Municipality                        Sta. Maria
 4: Government Branch                        Executive
 5: PE Organization Type                     Municipal Government
 6: PE Organization Type (Grouped)           Local Government Units (LGU)
 7: Bid Reference No.                        12433618
 8: Notice Title                             PB 2025-059 CONSTRUCTION OF ROAD SLOPE PROTECTION (ZONE 1, 2 & 3 BRGY. DUNGLAYAN)
 9: Classification                           Civil Works
10: Procurement Mode                         Public Bidding
11: Business Category                        Construction Projects
12: Funding Source                           Government of the Philippines (GOP)
13: Funding Instrument                       Bud